# KumoTabular (Nvidia) — Regression

Demonstrates **Kumo**, the zero-shot tabular foundation model, on the shared
retail/CPG regression datasets. Kumo is **self-hosted**: model weights are
downloaded from Hugging Face and inference runs locally in a single forward pass.

> **License note:** Kumo is released under a [commercially permissive license](https://openmdw.ai/license/1-1/) 

**Compute:** AIR Runtime (GPU Serverless) recommended with a single A10. CPU works but is slow.

**Prerequisite:** run [`shared/notebooks/00_data_preparation.ipynb`](../../../shared/notebooks/00_data_preparation.ipynb) first.

In [0]:
%pip install torch==2.8.0 git+https://github.com/NVIDIA/structured-data-models.git --quiet

In [0]:
dbutils.library.restartPython()

## Configuration

Catalog/schema must match the shared data-prep notebook. `common/` is added to the
path so we can import the shared evaluation helpers.

In [0]:
import os, sys

# Make the repo-level common/ package importable.
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))
COMMON_PATH = os.path.join(REPO_ROOT, "common")
if COMMON_PATH not in sys.path:
    sys.path.insert(0, COMMON_PATH)

from config import CATALOG

# Configure catalog and schema (must match shared/notebooks/00_data_preparation, imported from common/config.py)
SCHEMA = "default"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

# MLflow experiment (shared naming convention across vendors)
current_user = spark.sql("SELECT current_user()").collect()[0][0]
MLFLOW_EXPERIMENT_NAME = f"/Users/{current_user}/tabular-fm-databricks"

print(f"Catalog/schema: {CATALOG}.{SCHEMA}")

## Import libraries

In [0]:
import numpy as np
import pandas as pd
import mlflow

import torch
import sdm

from evaluation import (
    split_xy, regression_metrics, train_baselines_regression,
    log_result, RESULTS_TABLE,
)

mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

In [0]:
def evaluate_regression(table_name, target, task_name, test_size=0.2):
    df = spark.table(table_name).toPandas()
    X_train, X_test, y_train, y_test = split_xy(df, target=target, test_size=test_size)
    n_train, n_test, n_features = len(X_train), len(X_test), X_train.shape[1]

    with mlflow.start_run(run_name=f"{task_name}_kumo"):
        mlflow.log_params({
            "vendor": "kumo", "model_type": "Regressor",
            "task": task_name, "problem_type": "regression",
            "n_features": n_features, "train_samples": n_train, "test_samples": n_test,
        })
        device = "cuda" if torch.cuda.is_available() else "cpu"
        reg = sdm.models.KumoTabular(task="regression", device=device, size="large", pretrained=True)

        # Convert to SDM TableTensor format
        X_train_tensor = sdm.TableTensor.from_pandas(df=X_train, stypes=sdm.infer_stypes(X_train), device=device)
        y_train_tensor = sdm.TableTensor.from_pandas(df=y_train.to_frame(), stypes=sdm.infer_stypes(y_train.to_frame()), device=device)
        X_test_tensor = sdm.TableTensor.from_pandas(df=X_test, stypes=sdm.infer_stypes(X_test), device=device)

        reg.fit(x=X_train_tensor, y=y_train_tensor, num_estimators=8)
        # predict() returns 999 quantiles (q001–q999); use median (q500) as point estimate
        y_pred = reg.predict(X_test_tensor).to_pandas()["q500"].values
        metrics = regression_metrics(y_test, y_pred)
        mlflow.log_metrics({k: v for k, v in metrics.items() if v is not None})
        log_result(spark, vendor="kumo", task=task_name, problem_type="regression",
                   model_name="KumoTabularRegressor", metrics=metrics,
                   n_train=n_train, n_test=n_test, n_features=n_features)
    print(f"[{task_name}] Kumo: rmse={metrics['rmse']:.4f} "
          f"mae={metrics['mae']:.4f} r2={metrics['r2']:.4f}")

    for name, m in train_baselines_regression(X_train, y_train, X_test, y_test).items():
        log_result(spark, vendor="baseline", task=task_name, problem_type="regression",
                   model_name=name, metrics=m,
                   n_train=n_train, n_test=n_test, n_features=n_features)
        print(f"[{task_name}] {name}: rmse={m['rmse']:.4f} r2={m['r2']:.4f}")
    return metrics

## Price Elasticity

In [0]:
_ = evaluate_regression(
    table_name="price_elasticity_train",
    target="price_elasticity",
    task_name="price_elasticity",
)

## Supplier Lead Time

In [0]:
_ = evaluate_regression(
    table_name="supplier_lead_time_train",
    target="actual_lead_time_days",
    task_name="supplier_lead_time",
)

## Results

In [0]:
display(
    spark.table(RESULTS_TABLE)
         .where("problem_type = 'regression'")
         .orderBy("task", "vendor", "model_name")
)